# NexaTel Phase 3 — Churn KPI Calculations

**Prepared by:** Archana Arya  
**Project:** NexaTel Customer Churn Analysis  

## Objective
Calculate and validate the nine business KPIs specified in the project requirements.

In [1]:
##Confirming all source tables

import pandas as pd
import numpy as np

In [2]:
from pathlib import Path
data_folder = Path(r"C:\Users\Dell\Nexatel_project")
csv_files = sorted(data_folder.glob("*.csv"))
print("Number of files found :",len(csv_files))

for file in csv_files:
    print(file.name)



Number of files found : 24
billing.csv
cities.csv
complaints.csv
contracts.csv
customer_feedback.csv
customers.csv
data_quality_issue_log.csv
devices.csv
employees.csv
marketing_campaigns.csv
network_quality.csv
payments.csv
plan_history.csv
plans.csv
recharges.csv
regions.csv
retention_campaigns.csv
states.csv
stores.csv
subscriptions.csv
support_tickets.csv
usage_data.csv
usage_sms.csv
usage_voice.csv


In [3]:
#loading candidate KPI Table


candidate_kpi_tables = ["customers","subscriptions","plan_history","plans","billing","support_tickets","contracts"]

print(candidate_kpi_tables)

['customers', 'subscriptions', 'plan_history', 'plans', 'billing', 'support_tickets', 'contracts']


In [4]:
#validating the source for the first KPI:Customer Churn Rate.

#loading only customer table

customers = pd.read_csv(data_folder/"customers.csv")
print("Rows:",customers.shape[0])
print("Columns:",customers.shape[1])
print("column names:")
print(customers.columns.tolist())

customers.head()

Rows: 19076
Columns: 25
column names:
['customer_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'age', 'email', 'phone_number', 'address', 'city_id', 'city_name', 'state_id', 'pin_code', 'city_tier', 'region_id', 'occupation', 'annual_income_inr', 'customer_segment', 'product_line', 'plan_id', 'acquisition_channel', 'acquisition_date', 'tenure_months', 'customer_status', 'churn_date']


,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,...,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
0,C0000001,Harish,Shetty,Male,31-03-1988,38,harish.shetty909@yahoo.in,916647481156,"Plot 382, Tilak Lane, Raipur",CT0026,...,School Teacher,791058.0,Mass,Prepaid Mobile,PL002,Retail Store,15-01-2026,2,Churned,22-03-2026
1,C0000002,Anjali,Khan,Female,31-03-1974,52,anjali.khan525@rediffmail.com,918116536619,"Door No. 269, Civil Lines Colony, Gurugram",CT0009,...,Small Business Owner,1385499.0,Premium,Fiber Broadband,PL017,Retail Store,01-01-2014,146,Active,NaN
2,C0000003,Arjun,Hegde,Male,31-03-1978,48,arjun.hegde378@hotmail.com,918958153134,"House No. 320, Garden Nagar, Vadodara",CT0020,...,Auto/Cab Driver,394773.0,Enterprise,Enterprise,PL021,Tele-sales,11-11-2015,1,Churned,22-12-2015
3,C0000004,Vidya,Parmar,Female,31-03-1988,38,vidya.parmar387@hotmail.com,917763920098,"House No. 204, Industrial Street, Nagpur",CT0015,...,Operations Manager,1977532.0,Premium,Prepaid Mobile,PL006,Tele-sales,2014/10/18,137,Active,NaN
4,C0000005,Arjun,Saxena,Male,31-03-1988,38,arjun.saxena410@yahoo.in,916787478273,"Flat 156, Krishna Sector, Warangal",CT0043,...,Retired,847300.0,Value,Prepaid Mobile,PL003,Online,20-10-2014,108,Churned,22-10-2023


In [5]:
#grain validation

print("Total rows:", len(customers))

print("Unique customer IDs:",
    customers["customer_id"].nunique())

print("Duplicate customer IDs:",
    customers["customer_id"].duplicated().sum())

print("Missing customer IDs:",
    customers["customer_id"].isna().sum())

Total rows: 19076
Unique customer IDs: 19000
Duplicate customer IDs: 76
Missing customer IDs: 0


In [6]:
# grain is not clean yet total rows 19076 & unique is 19000 ,76 is duplicate
# inspect duplicate first

print(
    "Completely identical rows:",
    customers.duplicated().sum())

duplicate_customers = customers[customers["customer_id"].duplicated(keep=False)
].sort_values("customer_id")

print("Rows belonging to repeated customer IDs:",
    len(duplicate_customers))

duplicate_customers[[
        "customer_id",
        "plan_id",
        "acquisition_date",
        "customer_status",
        "churn_date"]].head(20)

Completely identical rows: 76
Rows belonging to repeated customer IDs: 152


,customer_id,plan_id,acquisition_date,customer_status,churn_date
97,C0000098,PL013,04-02-2021,Active,NaN
19044,C0000098,PL013,04-02-2021,Active,NaN
19074,C0000169,PL003,08-11-2020,Active,NaN
168,C0000169,PL003,08-11-2020,Active,NaN
19025,C0000784,PL002,27-09-2018,Active,NaN
783,C0000784,PL002,27-09-2018,Active,NaN
1391,C0001392,PL013,20-08-2024,Active,NaN
19051,C0001392,PL013,20-08-2024,Active,NaN
1393,C0001394,PL007,17-02-2022,Active,NaN
19020,C0001394,PL007,17-02-2022,Active,NaN


#We have confirmed:

76 completely identical duplicate rows
152 rows involved, meaning 76 customers appear twice
The repeated rows contain the same customer information

Therefore, it is safe to remove only the completely identical copies. We will preserve the original customers DataFrame and create a cleaned version.

In [7]:
customers_clean = customers.drop_duplicates().copy()

print("Rows after cleaning:", len(customers_clean))

print("Unique customer IDs:",
    customers_clean["customer_id"].nunique())

print("Duplicate customer IDs remaining:",
    customers_clean["customer_id"].duplicated().sum())

Rows after cleaning: 19000
Unique customer IDs: 19000
Duplicate customer IDs remaining: 0


In [8]:
#Final grain: one row represents one customer

## checking how churn is recording

print("Customer status values and counts:")

print(customers_clean["customer_status"].value_counts(
        dropna=False))

Customer status values and counts:
customer_status
Active     15494
Churned     3506
Name: count, dtype: int64


In [9]:
#check whether customer_status and churn_date agree

churned_mask = (customers_clean["customer_status"] == "Churned")

active_mask = (customers_clean["customer_status"] == "Active")

print("Churned customers without a churn date:",
    customers_clean.loc[
        churned_mask,
        "churn_date"].isna().sum())

print("Active customers with a churn date:",
    customers_clean.loc[
        active_mask,
        "churn_date"
    ].notna().sum())

Churned customers without a churn date: 0
Active customers with a churn date: 0


In [10]:
#validating date format and available time period.

customers_clean["acquisition_date_parsed"] = pd.to_datetime(
    customers_clean["acquisition_date"],format = "%d-%m-%Y",errors="coerce")

customers_clean["churn_date_parsed"] = pd.to_datetime(
customers_clean["churn_date"],format = "%d-%m-%Y",errors="coerce")

print(
    "Invalid acquisition dates:",
    customers_clean["acquisition_date_parsed"].isna().sum())

print("Invalid churn dates for churned customers:",
    customers_clean.loc[
        churned_mask,
        "churn_date_parsed"].isna().sum())


print("Acquisition date range:",
    customers_clean["acquisition_date_parsed"].min(),
    "to",
    customers_clean["acquisition_date_parsed"].max())


print("Churn date range:",
    customers_clean["churn_date_parsed"].min(),
    "to",
    customers_clean["churn_date_parsed"].max())

Invalid acquisition dates: 380
Invalid churn dates for churned customers: 0
Acquisition date range: 2014-01-01 00:00:00 to 2026-03-01 00:00:00
Churn date range: 2014-02-23 00:00:00 to 2026-03-30 00:00:00


In [11]:
#calculate missing and invalid acquisition date


missing_acquisition_dates = (customers_clean["acquisition_date"].isna().sum())

invalid_acquisition_rows = customers_clean[customers_clean["acquisition_date"].notna()
& customers_clean["acquisition_date_parsed"].isna()]

print("missing qriginal aquisition date:",missing_acquisition_dates)
print("non-missing but invalid acquisition date:",len(invalid_acquisition_rows))

invalid_acquisition_rows[["customer_id","acquisition_date","customer_status"]].head(10)

missing qriginal aquisition date: 0
non-missing but invalid acquisition date: 380


,customer_id,acquisition_date,customer_status
3,C0000004,2014/10/18,Active
33,C0000034,2023/04/16,Active
59,C0000060,2021/06/13,Active
74,C0000075,2024/12/13,Active
88,C0000089,2021/10/31,Active
94,C0000095,2018/06/01,Active
151,C0000152,2019/07/28,Active
308,C0000309,2016/09/03,Active
340,C0000341,2014/05/30,Active
347,C0000348,2014/01/01,Active


In [12]:
#lets check whether all 380 rows use the second format.

alternate_format_mask = (customers_clean["acquisition_date"]
    .astype(str)
    .str.fullmatch(r"\d{4}/\d{2}/\d{2}"))

print(
    "Rows using YYYY/MM/DD format:",
    alternate_format_mask.sum())

print("Unparsed rows not using YYYY/MM/DD:",
    (customers_clean["acquisition_date_parsed"].isna()
        & ~alternate_format_mask).sum())

Rows using YYYY/MM/DD format: 380
Unparsed rows not using YYYY/MM/DD: 0


In [13]:
#All 380 unparsed dates use the valid YYYY/MM/DD format, and no other unexpected format remains.

#Now convert only those 380 dates.


customers_clean.loc[
    alternate_format_mask,
    "acquisition_date_parsed"] = pd.to_datetime(
    customers_clean.loc[
        alternate_format_mask,
        "acquisition_date"],format="%Y/%m/%d",errors="coerce")

print("Unparsed acquisition dates remaining:",
    customers_clean["acquisition_date_parsed"].isna().sum())

print("Final acquisition date range:",
    customers_clean["acquisition_date_parsed"].min(),
    "to",customers_clean["acquisition_date_parsed"].max())

Unparsed acquisition dates remaining: 0
Final acquisition date range: 2014-01-01 00:00:00 to 2026-03-01 00:00:00


### All 19,000 acquisition dates are now successfully converted.

The project requirement confirms:

KPI period: Monthly
Formula: Customers lost during the month ÷ customers at the start of the month × 100
Target: Less than 2.1% monthly
No particular reporting month is specified, so we’ll calculate the KPI month by month.

Before constructing the monthly denominator, check that no customer churned before joining NexaTel.

In [14]:
date_order_issues = customers_clean[churned_mask
    &
    (customers_clean["churn_date_parsed"]
        <
        customers_clean["acquisition_date_parsed"])]

print(
    "Customers with churn date before acquisition date:",
    len(date_order_issues))

date_order_issues[
    ["customer_id",
        "acquisition_date",
        "churn_date"]].head()

Customers with churn date before acquisition date: 0


,customer_id,acquisition_date,churn_date


In [15]:
#March 2026 denominator and numerator

month_start = pd.Timestamp("2026-03-01")
next_month_start = pd.Timestamp("2026-04-01")

customers_at_start_mask = ((customers_clean["acquisition_date_parsed"]
        < month_start)
    &
    (customers_clean["churn_date_parsed"].isna()
        |
        (customers_clean["churn_date_parsed"]
            >= month_start)))

churned_during_month_mask = (
    (customers_clean["churn_date_parsed"]
        >= month_start)
    &
    (customers_clean["churn_date_parsed"]
        < next_month_start))

customers_lost_mask = (
    customers_at_start_mask
    &
    churned_during_month_mask)

customers_at_start = customers_at_start_mask.sum()
customers_lost = customers_lost_mask.sum()

print("Customers at the start of March 2026:",
    customers_at_start)

print("Starting customers lost during March 2026:",
    customers_lost)

Customers at the start of March 2026: 15688
Starting customers lost during March 2026: 196


In [16]:
#The validated March 2026 values are:

#Denominator — customers at start: 15,688
#Numerator — starting customers lost: 196

In [17]:
#Customer Churn Rate

march_2026_churn_rate = (customers_lost/customers_at_start) * 100

print("Customer Churn Rate for March 2026:",
    round(march_2026_churn_rate, 2),"%")

Customer Churn Rate for March 2026: 1.25 %


### KPI 1: Customer Churn Rate

- **Source table:** `customers.csv`
- **Cleaned DataFrame:** `customers_clean`
- **Grain:** One row per unique customer
- **Reporting period:** March 2026
- **Customers at the start of March:** 15,688
- **Starting customers lost during March:** 196
- **Formula:** Customers Lost ÷ Customers at Start × 100
- **Customer Churn Rate:** 1.25%
- **Target:** Less than 2.1% monthly
- **Target status:** Met

**Interpretation:** Approximately 1.25% of the customers who were active at the beginning of March 2026 churned during the month. The result is below NexaTel's monthly churn limit of 2.1%, so the churn target was achieved.

## KPI 2: Customer Retention Rate.

The validated source, grain, period, and denominator remain:

Source: customers.csv
DataFrame: customers_clean
Grain: One unique customer per row
Period: March 2026
Denominator: 15,688 customers at the start of March

The retention formula is:

Retention Rate=

(Customers at End−New Customers at End)/Customers at Start*100
	

In [18]:
# calculating required counts

customers_at_end_mask = (
    (customers_clean["acquisition_date_parsed"]
        < next_month_start) & (customers_clean["churn_date_parsed"].isna()
        |
        (customers_clean["churn_date_parsed"]>= next_month_start)))

new_customers_at_end_mask = (customers_at_end_mask
    &
    (customers_clean["acquisition_date_parsed"]>= month_start)
    &
    (customers_clean["acquisition_date_parsed"]< next_month_start))

customers_at_end = customers_at_end_mask.sum()
new_customers_at_end = new_customers_at_end_mask.sum()

retained_starting_customers = (customers_at_end - new_customers_at_end)

print("Customers at start:", customers_at_start)
print("Customers at end:", customers_at_end)
print("New customers included at end:", new_customers_at_end)
print("Retained starting customers:", retained_starting_customers)

Customers at start: 15688
Customers at end: 15494
New customers included at end: 2
Retained starting customers: 15492


In [19]:
# retention rate


march_2026_retention_rate = (retained_starting_customers
    /
    customers_at_start) * 100

print(
    "Customer Retention Rate for March 2026:",
    round(march_2026_retention_rate, 2),
    "%")

print("Churn Rate + Retention Rate:",
    round(march_2026_churn_rate + march_2026_retention_rate,2),"%")

Customer Retention Rate for March 2026: 98.75 %
Churn Rate + Retention Rate: 100.0 %


## KPI 2: Customer Retention Rate

**Definition:** The percentage of customers from the beginning of the month who remained with NexaTel through the end of the month.

**Source table:** `customers.csv`  
**Cleaned DataFrame:** `customers_clean`  
**Calculation grain:** One unique customer  
**Time period:** March 2026  

**Formula:**

Customer Retention Rate =  
((Customers at End − New Customers at End) / Customers at Start) × 100

**Validated values:**

- Customers at start: 15,688
- Customers at end: 15,494
- New customers included at end: 2
- Retained starting customers: 15,492

**Result:** 98.75%

**Validation:** Churn Rate (1.25%) + Retention Rate (98.75%) = 100%

**Business interpretation:** NexaTel retained 98.75% of the customers who were active at the beginning of March 2026. Only 1.25% of the starting customer base churned during the month.

KPI 3: ARPU.

The project formula is: ARPU = Recurring Revenue / Average Active Subscribers

Before calculating, we must identify:

The correct recurring-revenue column
Whether revenue includes GST
The billing table’s grain
The March 2026 billing records
The correct active-subscriber denominator

In [20]:
## lets load and inspect billing table


billing = pd.read_csv(data_folder / "billing.csv")

print("Billing rows:", billing.shape[0])
print("Billing columns:", billing.shape[1])

print("Billing column names:")
print(billing.columns.tolist())

billing.head()

Billing rows: 115313
Billing columns: 9
Billing column names:
['invoice_id', 'customer_id', 'billing_date', 'billing_period_month', 'base_amount_inr', 'gst_amount_inr', 'total_amount_inr', 'due_date', 'payment_status']


,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
0,INV00000001,C0000002,31-03-2026,03-2026,1490.77,268.34,1759.11,15-04-2026,Paid
1,INV00000002,C0000002,01-03-2026,03-2026,1396.28,251.33,1647.61,16-03-2026,Overdue
2,INV00000003,C0000002,30-01-2026,01-2026,1427.24,256.90,1684.14,14-02-2026,Paid
3,INV00000004,C0000002,31-12-2025,12-2025,1528.12,275.06,1803.18,15-01-2026,Paid
4,INV00000005,C0000002,01-12-2025,12-2025,1502.59,270.47,1773.06,16-12-2025,Paid


In [21]:
# validating invoice grain

print("Total billing rows:", len(billing))

print("Unique invoice IDs:",
    billing["invoice_id"].nunique())

print("Duplicate invoice IDs:",
    billing["invoice_id"].duplicated().sum())

print("Missing invoice IDs:",
    billing["invoice_id"].isna().sum())

print("Unique customers in billing:",
    billing["customer_id"].nunique())




Total billing rows: 115313
Unique invoice IDs: 114969
Duplicate invoice IDs: 344
Missing invoice IDs: 0
Unique customers in billing: 10169


In [22]:
#inspecting whether the repeated invoices are completely identical


print("Completely identical billing rows:",
    billing.duplicated().sum())

duplicate_invoices = billing[
    billing["invoice_id"].duplicated(
        keep=False)].sort_values("invoice_id")

print("Rows belonging to repeated invoice IDs:",
    len(duplicate_invoices))

duplicate_invoices[[
        "invoice_id",
        "customer_id",
        "billing_period_month",
        "base_amount_inr",
        "gst_amount_inr",
        "total_amount_inr",
        "payment_status"]].head(20)

Completely identical billing rows: 344
Rows belonging to repeated invoice IDs: 688


,invoice_id,customer_id,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,payment_status
127,INV00000128,C0000030,10-2025,562.30,101.21,663.51,Paid
115181,INV00000128,C0000030,10-2025,562.30,101.21,663.51,Paid
115278,INV00000652,C0000112,07-2025,552.95,99.53,652.48,Overdue
651,INV00000652,C0000112,07-2025,552.95,99.53,652.48,Overdue
1219,INV00001220,C0000232,01-2026,584.68,105.24,689.92,Overdue
115013,INV00001220,C0000232,01-2026,584.68,105.24,689.92,Overdue
1522,INV00001523,C0000272,07-2025,747.90,134.62,882.52,Paid
115081,INV00001523,C0000272,07-2025,747.90,134.62,882.52,Paid
1599,INV00001600,C0000282,01-2026,486.59,87.59,574.18,Overdue
115275,INV00001600,C0000282,01-2026,486.59,87.59,574.18,Overdue


In [23]:
#344 completely identical billing rows ,removing repeated invoices

billing_clean = billing.drop_duplicates().copy()

print("Billing rows after cleaning:",
    len(billing_clean))

print("Unique invoice IDs:",
    billing_clean["invoice_id"].nunique())

print("Duplicate invoice IDs remaining:",
    billing_clean["invoice_id"].duplicated().sum())

Billing rows after cleaning: 114969
Unique invoice IDs: 114969
Duplicate invoice IDs remaining: 0


In [24]:
#validate the billing period and March 2026 coverage


billing_clean["billing_month_parsed"] = pd.to_datetime(
    billing_clean["billing_period_month"],
    format="%m-%Y",
    errors="coerce")

print("Unparsed billing months:",
    billing_clean["billing_month_parsed"].isna().sum())

print("Billing month range:",
    billing_clean["billing_month_parsed"].min(),
    "to",
    billing_clean["billing_month_parsed"].max())

march_billing = billing_clean[
    billing_clean["billing_month_parsed"]
    == month_start]

print("March 2026 invoice rows:",
    len(march_billing))

print("March 2026 billed customers:",
    march_billing["customer_id"].nunique())

Unparsed billing months: 0
Billing month range: 2014-02-01 00:00:00 to 2026-03-01 00:00:00
March 2026 invoice rows: 16855
March 2026 billed customers: 8485


In [25]:
#March invoice count is nearly twice the customer count. This means many customers have more than one invoice in the same month


march_invoice_counts = (march_billing
    .groupby("customer_id")
    .size())

print("Invoices per customer in March 2026:")

print(march_invoice_counts
    .value_counts()
    .sort_index())

print("Maximum invoices for one customer:",
    march_invoice_counts.max())

Invoices per customer in March 2026:
1     115
2    8370
Name: count, dtype: int64
Maximum invoices for one customer: 2


In [26]:
#inspecting invoice

two_invoice_customers = march_invoice_counts[
    march_invoice_counts == 2].index

sample_customer_ids = two_invoice_customers[:5]

march_billing[march_billing["customer_id"].isin(
        sample_customer_ids)][[
        "invoice_id",
        "customer_id",
        "billing_date",
        "billing_period_month",
        "base_amount_inr",
        "gst_amount_inr",
        "total_amount_inr",
        "payment_status"]].sort_values(["customer_id", "billing_date"])

,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,payment_status
1,INV00000002,C0000002,01-03-2026,03-2026,1396.28,251.33,1647.61,Overdue
0,INV00000001,C0000002,31-03-2026,03-2026,1490.77,268.34,1759.11,Paid
14,INV00000015,C0000006,01-03-2026,03-2026,794.60,143.03,937.63,Overdue
13,INV00000014,C0000006,31-03-2026,03-2026,787.95,141.83,929.78,Paid
38,INV00000039,C0000009,01-03-2026,03-2026,904.90,162.88,1067.78,Unpaid
37,INV00000038,C0000009,31-03-2026,03-2026,868.70,156.37,1025.07,Unpaid
50,INV00000051,C0000012,01-03-2026,03-2026,489.05,88.03,577.08,Paid
49,INV00000050,C0000012,31-03-2026,03-2026,476.30,85.73,562.03,Unpaid
62,INV00000063,C0000014,01-03-2026,03-2026,991.40,178.45,1169.85,Overdue
61,INV00000062,C0000014,31-03-2026,03-2026,1021.20,183.82,1205.02,Paid


In [27]:
# inspecting plans table

plans = pd.read_csv(data_folder / "plans.csv")

print("Plan rows:", plans.shape[0])
print("Plan columns:", plans.shape[1])

print("Plan column names:")
print(plans.columns.tolist())

plans.head()

Plan rows: 25
Plan columns: 10
Plan column names:
['plan_id', 'plan_name', 'product_line', 'billing_cycle', 'monthly_charge', 'data_quota_gb', 'voice_minutes', 'sms_count', 'segment', 'is_active']


,plan_id,plan_name,product_line,billing_cycle,monthly_charge,data_quota_gb,voice_minutes,sms_count,segment,is_active
0,PL001,Prepaid Saver 199,Prepaid Mobile,Monthly,199,28,-1,300,Mass,Yes
1,PL002,Prepaid Smart 299,Prepaid Mobile,Monthly,299,56,-1,-1,Mass,Yes
2,PL003,Prepaid Plus 399,Prepaid Mobile,Monthly,399,84,-1,-1,Value,Yes
3,PL004,Prepaid Data 449,Prepaid Mobile,Monthly,449,112,-1,-1,Value,Yes
4,PL005,Prepaid Annual 2999,Prepaid Mobile,Annual,250,730,-1,-1,Value,Yes


In [28]:
print("Total plan rows:", len(plans))

print("Unique plan IDs:",
    plans["plan_id"].nunique())

print("Duplicate plan IDs:",
    plans["plan_id"].duplicated().sum())

print("Missing monthly charges:",
    plans["monthly_charge"].isna().sum())

print("Zero or negative monthly charges:",
    (plans["monthly_charge"] <= 0).sum())

print("\nBilling cycles:")
print(plans["billing_cycle"].value_counts(
        dropna=False))

print("\nPlan active values:")
print(plans["is_active"].value_counts(
        dropna=False))

Total plan rows: 25
Unique plan IDs: 25
Duplicate plan IDs: 0
Missing monthly charges: 0
Zero or negative monthly charges: 0

Billing cycles:
billing_cycle
Monthly    19
Annual      6
Name: count, dtype: int64

Plan active values:
is_active
Yes    25
Name: count, dtype: int64


In [29]:
#inspecting subscription table

subscriptions = pd.read_csv(
    data_folder / "subscriptions.csv")

print("Subscription rows:",
    subscriptions.shape[0])

print("Subscription columns:",
    subscriptions.shape[1])

print("Subscription column names:")
print(subscriptions.columns.tolist())

subscriptions.head()

Subscription rows: 20523
Subscription columns: 8
Subscription column names:
['subscription_id', 'customer_id', 'plan_id', 'start_date', 'end_date', 'billing_cycle', 'monthly_charge_inr', 'subscription_status']


,subscription_id,customer_id,plan_id,start_date,end_date,billing_cycle,monthly_charge_inr,subscription_status
0,S0000001,C0000001,PL002,15-01-2026,22-03-2026,Monthly,299.0,Terminated
1,S0000002,C0000002,PL017,01-01-2014,NaN,Monthly,1499.0,Active
2,S0000003,C0000003,PL021,11-11-2015,22-12-2015,Annual,4166.0,Terminated
3,S0000004,C0000004,PL006,18-10-2014,NaN,Annual,300.0,Active
4,S0000005,C0000005,PL003,20-10-2014,22-10-2023,Monthly,399.0,Terminated


In [30]:
#validating its grain and required fields


print("Total subscription rows:",
    len(subscriptions))

print("Unique subscription IDs:",
    subscriptions["subscription_id"].nunique())

print("Duplicate subscription IDs:",
    subscriptions["subscription_id"].duplicated().sum())

print("Missing subscription IDs:",
    subscriptions["subscription_id"].isna().sum())

print("Unique subscription customers:",
    subscriptions["customer_id"].nunique())

print("Missing plan IDs:",
    subscriptions["plan_id"].isna().sum())

print("\nSubscription status values:")

print(subscriptions["subscription_status"].value_counts(dropna=False))

Total subscription rows: 20523
Unique subscription IDs: 20523
Duplicate subscription IDs: 0
Missing subscription IDs: 0
Unique subscription customers: 19000
Missing plan IDs: 41

Subscription status values:
subscription_status
Active           16930
Terminated        3491
UNKNOWN_STATE      102
Name: count, dtype: int64


In [31]:
#validating start_date and end_date

subscriptions_clean = subscriptions.copy()

subscriptions_clean["start_date_parsed"] = pd.to_datetime(
    subscriptions_clean["start_date"],
    format="%d-%m-%Y",
    errors="coerce")

subscriptions_clean["end_date_parsed"] = pd.to_datetime(
    subscriptions_clean["end_date"],
    format="%d-%m-%Y",
    errors="coerce")

print("Missing original start dates:",
    subscriptions_clean["start_date"].isna().sum())

print("Unparsed start dates:",
    subscriptions_clean["start_date_parsed"].isna().sum())

print("Non-missing but unparsed end dates:",
    (subscriptions_clean["end_date"].notna()&
        subscriptions_clean["end_date_parsed"].isna()).sum())

print("Subscription start-date range:",
    subscriptions_clean["start_date_parsed"].min(),
    "to",
    subscriptions_clean["start_date_parsed"].max())

print("Subscription end-date range:",
    subscriptions_clean["end_date_parsed"].min(),
    "to",
    subscriptions_clean["end_date_parsed"].max())

Missing original start dates: 0
Unparsed start dates: 0
Non-missing but unparsed end dates: 0
Subscription start-date range: 2014-01-01 00:00:00 to 2027-03-11 00:00:00
Subscription end-date range: 2014-02-23 00:00:00 to 2026-03-30 00:00:00


In [32]:
# verifying date issue 

future_start_mask = (subscriptions_clean["start_date_parsed"]
    >= next_month_start)

end_before_start_mask = (subscriptions_clean["end_date_parsed"].notna()
    &
    (subscriptions_clean["end_date_parsed"]
        <
        subscriptions_clean["start_date_parsed"]))

print("Subscriptions starting after March 2026:",
    future_start_mask.sum())

print("Subscriptions ending before their start date:",
    end_before_start_mask.sum())

subscriptions_clean.loc[
    future_start_mask,
    [
        "subscription_id",
        "customer_id",
        "start_date",
        "end_date",
        "subscription_status"]].head(10)

Subscriptions starting after March 2026: 52
Subscriptions ending before their start date: 0


,subscription_id,customer_id,start_date,end_date,subscription_status
19015,S0019016,C0000223,12-04-2026,NaN,Active
19018,S0019019,C0000320,17-01-2027,NaN,Active
19044,S0019045,C0000570,02-01-2027,NaN,Active
19064,S0019065,C0000805,13-10-2026,NaN,Active
19072,S0019073,C0000890,05-06-2026,NaN,Active
19078,S0019079,C0000954,09-06-2026,NaN,Active
19108,S0019109,C0001237,06-06-2026,NaN,Active
19110,S0019111,C0001268,24-10-2026,NaN,Active
19132,S0019133,C0001653,10-06-2026,NaN,Active
19135,S0019136,C0001689,17-05-2026,NaN,Active


In [33]:
active_subscription_mask = (subscriptions_clean["subscription_status"]
    == "Active")

terminated_subscription_mask = (
    subscriptions_clean["subscription_status"]
    == "Terminated")

unknown_subscription_mask = (
    subscriptions_clean["subscription_status"]
    == "UNKNOWN_STATE")

print("Active subscriptions with an end date:",
    subscriptions_clean.loc[
        active_subscription_mask,
        "end_date_parsed"
    ].notna().sum())

print("Terminated subscriptions without an end date:",
    subscriptions_clean.loc[
        terminated_subscription_mask,
        "end_date_parsed"
    ].isna().sum())

print("Unknown subscriptions without an end date:",
    subscriptions_clean.loc[
        unknown_subscription_mask,
        "end_date_parsed"
    ].isna().sum())

print("Unknown subscriptions with an end date:",
    subscriptions_clean.loc[
        unknown_subscription_mask,
        "end_date_parsed"
    ].notna().sum())

Active subscriptions with an end date: 0
Terminated subscriptions without an end date: 0
Unknown subscriptions without an end date: 87
Unknown subscriptions with an end date: 15


In [34]:
# validate monthly_charge_inr


print("Missing monthly charges:",
    subscriptions_clean[
        "monthly_charge_inr"
    ].isna().sum())

print("Zero or negative monthly charges:",
    (subscriptions_clean[
            "monthly_charge_inr"] <= 0).sum())

print("Minimum monthly charge:",
    subscriptions_clean[
        "monthly_charge_inr"
    ].min())

print("Maximum monthly charge:",
    subscriptions_clean[
        "monthly_charge_inr"
    ].max())

print("Missing-plan rows with an available charge:",
    subscriptions_clean.loc[
        subscriptions_clean["plan_id"].isna(),
        "monthly_charge_inr"].notna().sum())

Missing monthly charges: 0
Zero or negative monthly charges: 0
Minimum monthly charge: 99.0
Maximum monthly charge: 8333.0
Missing-plan rows with an available charge: 41


In [56]:
# ARPU Calculations


opening_active = (
    (subscriptions_clean["start_date_parsed"] < month_start)
    &
    (
        subscriptions_clean["end_date_parsed"].isna()
        |
        (subscriptions_clean["end_date_parsed"] >= month_start)
    )
)

closing_active = (
    (subscriptions_clean["start_date_parsed"] < next_month_start)
    &
    (subscriptions_clean["end_date_parsed"].isna()
        |
        (subscriptions_clean["end_date_parsed"] >= next_month_start)))

opening_subscribers = opening_active.sum()
closing_subscribers = closing_active.sum()

average_active_subscribers = (
    opening_subscribers + closing_subscribers) / 2

opening_recurring_revenue = subscriptions_clean.loc[
    opening_active, "monthly_charge_inr"].sum()

closing_recurring_revenue = subscriptions_clean.loc[
    closing_active, "monthly_charge_inr"].sum()

average_recurring_revenue = (
    opening_recurring_revenue
    +
    closing_recurring_revenue) / 2

march_2026_arpu = (average_recurring_revenue
    /
    average_active_subscribers)

print("Opening active subscribers:", opening_subscribers)
print("Closing active subscribers:", closing_subscribers)
print("Average active subscribers:", average_active_subscribers)

print("Average recurring revenue (INR):",round(average_recurring_revenue, 2))

print("Average Revenue per Subscription (INR):",
    round(march_2026_arpu, 2))

Opening active subscribers: 17151
Closing active subscribers: 16965
Average active subscribers: 17058.0
Average recurring revenue (INR): 15029176.0
Average Revenue per Subscription (INR): 881.06


## KPI 3: Average Revenue per User (ARPU)

**Source:** `subscriptions.csv`  
**Revenue field:** `monthly_charge_inr`  
**Numerator grain:** Active subscription charges  
**Denominator grain:** Unique active customers  
**Period:** March 2026  

**Formula:**  
ARPU = Average Recurring Revenue / Average Active Customers

- Opening active customers: 15,913
- Closing active customers: 15,731
- Average active customers: 15,822
- Average recurring revenue: ₹15,029,176
- **ARPU: ₹949.89**

**Interpretation:** NexaTel generated an average monthly recurring revenue of ₹949.89 per active customer during March 2026.

## KPI 4: Customer Lifetime Value (CLV)

**Formula:**

CLV = ARPU × Gross Margin % × (1 / Monthly Churn Rate)

**Available inputs:**

- March 2026 ARPU: ₹881.06
- March 2026 churn rate: 1.25%
- Gross-margin percentage: Not provided

**Status:** Pending business input

**Data limitation:** The NexaTel PRD and available source tables do not provide a gross-margin percentage or direct service-cost data. Therefore, a reliable CLV cannot be calculated without introducing an unsupported assumption.

**Required action:** Obtain the approved gross-margin percentage from the business owner or project mentor before finalizing this KPI.

In [59]:
## KPI5

#VALIDATING REQUIRED COUNTS


opening_active_customers = subscriptions_clean.loc[
    opening_active,
    "customer_id"
].nunique()

closing_active_customers = subscriptions_clean.loc[
    closing_active,
    "customer_id"
].nunique()

average_active_customers = (
    opening_active_customers
    +
    closing_active_customers
) / 2

march_2026_arpu = (
    average_recurring_revenue
    /
    average_active_customers
)

print(
    "Opening active customers:",
    opening_active_customers
)

print(
    "Closing active customers:",
    closing_active_customers
)

print(
    "Average active customers:",
    average_active_customers
)

print(
    "Corrected ARPU for March 2026 (INR):",
    round(march_2026_arpu, 2)
)

Opening active customers: 15913
Closing active customers: 15731
Average active customers: 15822.0
Corrected ARPU for March 2026 (INR): 949.89


In [60]:
#KPI5 #Revenue lost to churn


march_2026_revenue_lost = (
    customers_lost
    *
    march_2026_arpu)

print("Revenue Lost to Churn in March 2026 (INR):",
    round(march_2026_revenue_lost, 2))

Revenue Lost to Churn in March 2026 (INR): 186178.64


## KPI 5: Revenue Lost to Churn

**Source:** `customers.csv` and `subscriptions.csv`  
**Grain:** Unique customer  
**Period:** March 2026  

**Formula:**  
Revenue Lost to Churn = Churned Customers × ARPU

- Churned starting customers: 196
- ARPU: ₹949.89
- **Revenue lost: ₹186,178.64**

**Interpretation:** NexaTel lost an estimated ₹186,178.64 in monthly recurring revenue due to customers who churned during March 2026.

In [38]:
# KPI 6 Monthly Recurring Revenue

march_2026_mrr = (subscriptions_clean.loc[
        closing_active,
        "monthly_charge_inr"].sum())

print("Monthly Recurring Revenue for March 2026 (INR):",
    round(march_2026_mrr, 2))

Monthly Recurring Revenue for March 2026 (INR): 14971268.0


## KPI 6: Monthly Recurring Revenue (MRR)

**Source:** `subscriptions.csv`  
**Revenue field:** `monthly_charge_inr`  
**Grain:** One active subscription  
**Measurement date:** End of March 2026  

**Formula:**  
MRR = Sum of recurring monthly charges for active subscriptions

- Closing active subscriptions: 16,965
- **MRR: ₹14,971,268**

**Interpretation:** NexaTel had approximately ₹1.50 crore in monthly recurring revenue from subscriptions active at the end of March 2026.

In [39]:
#KPI 7: First Contact Resolution

#The formula is: FCR=Tickets resolved on first contact/Total eligible support tickets
	# ×100

In [40]:
# inspecting support_tickets


support_tickets = pd.read_csv(
    data_folder / "support_tickets.csv")

print("Support ticket rows:",
    support_tickets.shape[0])

print("Support ticket columns:",
    support_tickets.shape[1])

print("Column names:")
print(support_tickets.columns.tolist())

support_tickets.head()

Support ticket rows: 31490
Support ticket columns: 11
Column names:
['ticket_id', 'customer_id', 'created_date', 'category', 'priority', 'channel', 'status', 'first_contact_resolution', 'assigned_employee_id', 'resolution_hours', 'resolution_date']


,ticket_id,customer_id,created_date,category,priority,channel,status,first_contact_resolution,assigned_employee_id,resolution_hours,resolution_date
0,TK00000001,C0000001,08-09-2017,Billing Dispute,Medium,Store,Pending,No,EM00222,NaN,NaN
1,TK00000002,C0000001,12-01-2022,Slow Data Speed,Medium,Chat,Resolved,Yes,EM00466,25.6,13-01-2022
2,TK00000003,C0000002,21-09-2015,Network Issue,Low,Store,Pending,No,EM00462,NaN,NaN
3,TK00000004,C0000002,02-10-2020,Plan Change,Low,App,Resolved,Yes,EM00117,17.8,03-10-2020
4,TK00000005,C0000002,03-10-2021,Plan Change,Low,App,Resolved,Yes,EM00034,14.1,04-10-2021


In [41]:
print(
    "Duplicate ticket IDs:",
    support_tickets["ticket_id"].duplicated().sum())

support_tickets_clean = (
    support_tickets
    .drop_duplicates()
    .copy())

support_tickets_clean["created_date_parsed"] = pd.to_datetime(
    support_tickets_clean["created_date"],
    format="%d-%m-%Y",
    errors="coerce")

march_tickets = support_tickets_clean[
    (support_tickets_clean["created_date_parsed"]
        >= month_start)
    &
    (support_tickets_clean["created_date_parsed"]
        < next_month_start)]

print("Unparsed created dates:",
    support_tickets_clean["created_date_parsed"].isna().sum())

print("March 2026 tickets:",
    len(march_tickets))

print("\nMarch ticket statuses:")
print(march_tickets["status"].value_counts(
        dropna=False))

print("\nMarch first-contact-resolution values:")
print(march_tickets["first_contact_resolution"].value_counts(
        dropna=False))

Duplicate ticket IDs: 0
Unparsed created dates: 0
March 2026 tickets: 212

March ticket statuses:
status
Resolved     128
Pending       56
Escalated     15
Reopened      13
Name: count, dtype: int64

March first-contact-resolution values:
first_contact_resolution
No     124
Yes     88
Name: count, dtype: int64


In [42]:
first_contact_resolved = (
    march_tickets[
        "first_contact_resolution"
    ] == "Yes").sum()

total_march_tickets = len(march_tickets)

march_2026_fcr = (
    first_contact_resolved
    /
    total_march_tickets) * 100

print(
    "First Contact Resolution Rate:",
    round(march_2026_fcr, 2),
    "%")

if march_2026_fcr > 78:
    print("Target status: Met")
else:
    print("Target status: Not Met")

First Contact Resolution Rate: 41.51 %
Target status: Not Met


## KPI 7: First Contact Resolution (FCR)

**Source:** `support_tickets.csv`  
**Grain:** One support ticket  
**Period:** March 2026  

**Formula:**  
FCR = Tickets Resolved on First Contact / Total Tickets × 100

- Total March tickets: 212
- Resolved on first contact: 88
- **FCR: 41.51%**
- Target: Greater than 78%
- **Target status: Not Met**

**Interpretation:** Only 41.51% of March support tickets were resolved on the first contact. NexaTel should investigate support processes, agent training and escalation causes to improve FCR.

In [43]:
#KPI 8: Average Resolution Time


march_tickets = march_tickets.copy()

march_tickets["resolution_hours_numeric"] = pd.to_numeric(
    march_tickets["resolution_hours"],
    errors="coerce")

resolved_march_tickets = march_tickets[
    (march_tickets["status"] == "Resolved")
    &
    (march_tickets["resolution_hours_numeric"]
        >= 0)]

average_resolution_time = resolved_march_tickets[
    "resolution_hours_numeric"].mean()

print("Eligible resolved March tickets:",
    len(resolved_march_tickets))

print("Average Resolution Time:",
    round(average_resolution_time, 2),
    "hours")

if average_resolution_time < 24:
    print("Target status: Met")
else:
    print("Target status: Not Met")

Eligible resolved March tickets: 128
Average Resolution Time: 15.21 hours
Target status: Met


## KPI 8: Average Resolution Time

**Source:** `support_tickets.csv`  
**Grain:** One resolved support ticket  
**Period:** March 2026  
**Denominator:** 128 resolved March tickets with valid resolution hours  

**Formula:**  
Average Resolution Time = Mean resolution hours of resolved tickets

- Eligible resolved tickets: 128
- **Average resolution time: 15.21 hours**
- Target: Less than 24 hours
- **Target status: Met**

**Interpretation:** NexaTel resolved March support tickets in an average of 15.21 hours, which is 8.79 hours below the maximum target of 24 hours.

In [44]:
# KPI -9 Contract Renewal Rate

#he formula is:

#Renewal Rate=Contracts Renewed/Contracts Due for Renewal *100

#INSPECTING CONTRACTS TABLE


contracts = pd.read_csv(
    data_folder / "contracts.csv")

print("Contract rows:",
    contracts.shape[0])

print("Contract columns:",
    contracts.shape[1])

print("Contract column names:")
print(contracts.columns.tolist())

contracts.head()

Contract rows: 10607
Contract columns: 9
Contract column names:
['contract_id', 'customer_id', 'plan_id', 'contract_type', 'contract_length_months', 'start_date', 'end_date', 'auto_renew', 'renewal_status']


,contract_id,customer_id,plan_id,contract_type,contract_length_months,start_date,end_date,auto_renew,renewal_status
0,CN000001,C0000002,PL017,Fiber Broadband,24,01-01-2014,22-12-2015,Yes,Renewed
1,CN000002,C0000003,PL021,Enterprise,18,11-11-2015,04-05-2017,Yes,Not Renewed
2,CN000003,C0000004,PL006,Prepaid Mobile,12,18-10-2014,13-10-2015,No,Renewed
3,CN000004,C0000006,PL025,Smart Home,12,20-06-2018,15-06-2019,Yes,Renewed
4,CN000005,C0000007,PL007,Postpaid Mobile,24,13-01-2024,02-01-2026,Yes,Cancelled


In [45]:
print("Duplicate contract IDs:",
    contracts["contract_id"].duplicated().sum())

contracts_clean = (
    contracts
    .drop_duplicates()
    .copy())

contracts_clean["end_date_parsed"] = pd.to_datetime(
    contracts_clean["end_date"],
    format="%d-%m-%Y",
    errors="coerce")

print("Unparsed contract end dates:",
    contracts_clean["end_date_parsed"].isna().sum())

print("\nContract types:")
print(contracts_clean["contract_type"].value_counts(
        dropna=False))

print("\nRenewal status values:")
print(contracts_clean["renewal_status"].value_counts(dropna=False))

Duplicate contract IDs: 0
Unparsed contract end dates: 0

Contract types:
contract_type
Postpaid Mobile    3753
Prepaid Mobile     2904
Fiber Broadband    2193
Enterprise          811
IoT Solutions       579
Smart Home          367
Name: count, dtype: int64

Renewal status values:
renewal_status
Renewed         3676
Auto-Renewed    2266
Active          1829
Pending         1360
Not Renewed     1058
Cancelled        418
Name: count, dtype: int64


In [46]:
eligible_contract_types = [
    "Postpaid Mobile",
    "Fiber Broadband"]

march_contracts_due = contracts_clean[
    (contracts_clean["contract_type"].isin(
            eligible_contract_types))
    &
    (contracts_clean["end_date_parsed"]
        >= month_start)
    &
    (contracts_clean["end_date_parsed"]
        < next_month_start)].copy()

renewed_contracts = march_contracts_due[
    "renewal_status"].isin(["Renewed", "Auto-Renewed"]).sum()

contracts_due = len(march_contracts_due)

march_2026_renewal_rate = (
    renewed_contracts
    /
    contracts_due) * 100

print("Contracts due for renewal:", contracts_due)
print("Contracts renewed:", renewed_contracts)

print("Contract Renewal Rate:",
    round(march_2026_renewal_rate, 2),
    "%")

if march_2026_renewal_rate > 88:
    print("Target status: Met")
else:
    print("Target status: Not Met")

Contracts due for renewal: 44
Contracts renewed: 26
Contract Renewal Rate: 59.09 %
Target status: Not Met


## KPI 9: Contract Renewal Rate

**Source:** `contracts.csv`  
**Grain:** One contract  
**Period:** March 2026  
**Eligible contract types:** Postpaid Mobile and Fiber Broadband  
**Renewed statuses:** Renewed and Auto-Renewed  

**Formula:**  
Contract Renewal Rate = Renewed Contracts / Contracts Due for Renewal × 100

- Contracts due for renewal: 44
- Contracts renewed: 26
- **Contract renewal rate: 59.09%**
- Target: Greater than 88%
- **Target status: Not Met**

**Interpretation:** NexaTel renewed 59.09% of eligible Postpaid Mobile and Fiber Broadband contracts due in March 2026. This was 28.91 percentage points below the target.

In [47]:
#KPI 10 - Average Customer Tenure.

active_customer_base = customers_clean[
    customers_clean["customer_status"] == "Active"].copy()

active_customer_base["tenure_months_numeric"] = pd.to_numeric(
    active_customer_base["tenure_months"],
    errors="coerce")

print("Active customers:",
    len(active_customer_base))

print("Missing or invalid tenure values:",
    active_customer_base[
        "tenure_months_numeric"].isna().sum())

print("Negative tenure values:",
    (active_customer_base[
            "tenure_months_numeric"] < 0).sum())

average_customer_tenure = active_customer_base[
    "tenure_months_numeric"].mean()

print("Average Customer Tenure:",
    round(average_customer_tenure, 2),"months")

Active customers: 15494
Missing or invalid tenure values: 0
Negative tenure values: 0
Average Customer Tenure: 87.08 months


## KPI 10: Average Customer Tenure

**Source:** `customers.csv`  
**Cleaned DataFrame:** `customers_clean`  
**Grain:** One unique active customer  
**Measurement period:** End of March 2026  

**Formula:**  
Average Customer Tenure = Mean tenure_months of the active customer base

- Active customers: 15,494
- Missing or invalid tenure values: 0
- Negative tenure values: 0
- **Average customer tenure: 87.08 months**
- Equivalent to approximately 7.26 years

**Interpretation:** NexaTel's active customers have remained with the company for an average of approximately 7.26 years. A rising average tenure generally indicates stronger customer loyalty and stickiness.

In [48]:
#Final KPI'S Table


kpi_summary = pd.DataFrame({
    "KPI": [
        "Customer Churn Rate",
        "Customer Retention Rate",
        "ARPU",
        "Customer Lifetime Value",
        "Revenue Lost to Churn",
        "Monthly Recurring Revenue",
        "First Contact Resolution",
        "Average Resolution Time",
        "Contract Renewal Rate",
        "Average Customer Tenure"],

    "March 2026 Result": [
        f"{march_2026_churn_rate:.2f}%",
        f"{march_2026_retention_rate:.2f}%",
        f"₹{march_2026_arpu:,.2f}",
        "Pending",
        f"₹{march_2026_revenue_lost:,.2f}",
        f"₹{march_2026_mrr:,.2f}",
        f"{march_2026_fcr:.2f}%",
        f"{average_resolution_time:.2f} hours",
        f"{march_2026_renewal_rate:.2f}%",
        f"{average_customer_tenure:.2f} months"],

    "Target": [
        "< 2.1%",
        "Not specified",
        "Not specified",
        "Not available",
        "Not specified",
        "Not specified",
        "> 78%",
        "< 24 hours",
        "> 88%",
        "Rising trend"],

    "Status": [
        "Met",
        "Reported",
        "Reported",
        "Gross margin required",
        "Reported",
        "Reported",
        "Not Met",
        "Met",
        "Not Met",
        "Baseline"]})

kpi_summary

,KPI,March 2026 Result,Target,Status
0,Customer Churn Rate,1.25%,< 2.1%,Met
1,Customer Retention Rate,98.75%,Not specified,Reported
2,ARPU,₹949.89,Not specified,Reported
3,Customer Lifetime Value,Pending,Not available,Gross margin required
4,Revenue Lost to Churn,"₹186,178.64",Not specified,Reported
5,Monthly Recurring Revenue,"₹14,971,268.00",Not specified,Reported
6,First Contact Resolution,41.51%,> 78%,Not Met
7,Average Resolution Time,15.21 hours,< 24 hours,Met
8,Contract Renewal Rate,59.09%,> 88%,Not Met
9,Average Customer Tenure,87.08 months,Rising trend,Baseline


# Optional Bonus — Cohort Retention Analysis

Customers are grouped by their acquisition month. Retention will be measured after 3, 6 and 12 months for each cohort.

In [49]:
cohort_data = customers_clean[
    ["customer_id",
        "acquisition_date_parsed",
        "churn_date_parsed"]].copy()

cohort_data["acquisition_cohort"] = (
    cohort_data["acquisition_date_parsed"]
    .dt.to_period("M"))

print("Missing acquisition cohorts:",
    cohort_data["acquisition_cohort"].isna().sum())

print("Earliest cohort:",
    cohort_data["acquisition_cohort"].min())

print("Latest cohort:",cohort_data["acquisition_cohort"].max())

cohort_data.head()

Missing acquisition cohorts: 0
Earliest cohort: 2014-01
Latest cohort: 2026-03


,customer_id,acquisition_date_parsed,churn_date_parsed,acquisition_cohort
0,C0000001,2026-01-15,2026-03-22,2026-01
1,C0000002,2014-01-01,NaT,2014-01
2,C0000003,2015-11-11,2015-12-22,2015-11
3,C0000004,2014-10-18,NaT,2014-10
4,C0000005,2014-10-20,2023-10-22,2014-10


In [50]:
cohort_cutoff = pd.Timestamp("2026-03-31")

cohort_data["milestone_3m"] = (
    cohort_data["acquisition_date_parsed"]
    + pd.DateOffset(months=3))

cohort_data["milestone_6m"] = (
    cohort_data["acquisition_date_parsed"]
    + pd.DateOffset(months=6))

cohort_data["milestone_12m"] = (
    cohort_data["acquisition_date_parsed"]
    + pd.DateOffset(months=12))

cohort_data["eligible_3m"] = (
    cohort_data["milestone_3m"]
    <= cohort_cutoff)

cohort_data["eligible_6m"] = (
    cohort_data["milestone_6m"]
    <= cohort_cutoff)

cohort_data["eligible_12m"] = (
    cohort_data["milestone_12m"]
    <= cohort_cutoff)

print("Customers eligible for 3-month retention:",
    cohort_data["eligible_3m"].sum())

print("Customers eligible for 6-month retention:",
    cohort_data["eligible_6m"].sum())

print("Customers eligible for 12-month retention:",
    cohort_data["eligible_12m"].sum())

cohort_data[[
        "customer_id",
        "acquisition_cohort",
        "milestone_3m",
        "milestone_6m",
        "milestone_12m"
    ]].head()

Customers eligible for 3-month retention: 18793
Customers eligible for 6-month retention: 18439
Customers eligible for 12-month retention: 17756


,customer_id,acquisition_cohort,milestone_3m,milestone_6m,milestone_12m
0,C0000001,2026-01,2026-04-15,2026-07-15,2027-01-15
1,C0000002,2014-01,2014-04-01,2014-07-01,2015-01-01
2,C0000003,2015-11,2016-02-11,2016-05-11,2016-11-11
3,C0000004,2014-10,2015-01-18,2015-04-18,2015-10-18
4,C0000005,2014-10,2015-01-20,2015-04-20,2015-10-20


In [51]:

cohort_data["retained_3m"] = np.where(
    cohort_data["eligible_3m"],
    (cohort_data["churn_date_parsed"].isna()
        |
        (cohort_data["churn_date_parsed"]
            >= cohort_data["milestone_3m"])
    ).astype(int),
    np.nan)

cohort_data["retained_6m"] = np.where(
    cohort_data["eligible_6m"],
    (cohort_data["churn_date_parsed"].isna()
        |
        (cohort_data["churn_date_parsed"]
            >= cohort_data["milestone_6m"])
    ).astype(int),
    np.nan)

cohort_data["retained_12m"] = np.where(
    cohort_data["eligible_12m"],
    (cohort_data["churn_date_parsed"].isna()
        |
        (cohort_data["churn_date_parsed"]
            >= cohort_data["milestone_12m"]
        )).astype(int),np.nan)

print("3-month retention flags:")
print(cohort_data["retained_3m"].value_counts(dropna=False))

print("\n6-month retention flags:")
print(cohort_data["retained_6m"].value_counts(dropna=False))

print("\n12-month retention flags:")
print(cohort_data["retained_12m"].value_counts(dropna=False))

3-month retention flags:
retained_3m
1.0    18451
0.0      342
NaN      207
Name: count, dtype: int64

6-month retention flags:
retained_6m
1.0    17882
NaN      561
0.0      557
Name: count, dtype: int64

12-month retention flags:
retained_12m
1.0    16961
NaN     1244
0.0      795
Name: count, dtype: int64


In [52]:
cohort_retention = pd.pivot_table(
    cohort_data,
    index="acquisition_cohort",
    values=[
        "retained_3m",
        "retained_6m",
        "retained_12m"],aggfunc="mean") * 100

cohort_retention = cohort_retention[["retained_3m",
        "retained_6m",
        "retained_12m"]]

cohort_retention.columns = [
    "3-Month Retention (%)",
    "6-Month Retention (%)",
    "12-Month Retention (%)"]

cohort_retention = cohort_retention.round(2)

all_cohorts = (
    cohort_data["acquisition_cohort"]
    .sort_values()
    .unique())

cohort_retention = cohort_retention.reindex(
    all_cohorts)

cohort_retention.tail(15)

,3-Month Retention (%),6-Month Retention (%),12-Month Retention (%)
acquisition_cohort,,,
2025-01,96.26,85.98,68.22
2025-02,90.27,81.42,66.37
2025-03,91.74,84.30,71.90
2025-04,92.93,84.85,NaN
2025-05,94.69,86.73,NaN
2025-06,79.63,74.07,NaN
2025-07,88.00,76.80,NaN
2025-08,89.60,73.60,NaN
2025-09,81.42,65.49,NaN


## Bonus Findings — Cohort Retention

- Retention generally declines as the measurement period increases from 3 to 6 and 12 months.
- The January 2025 cohort declined from 96.26% retention after 3 months to 68.22% after 12 months.
- Among the 2025 cohorts with complete 12-month observations, March 2025 had the strongest 12-month retention at 71.90%.
- The June 2025 cohort had lower early retention: 79.63% after 3 months and 74.07% after 6 months.
- `NaN` does not mean zero retention. It means the cohort had not completed the required observation period by 31 March 2026.

**Business interpretation:** NexaTel should investigate cohorts with weaker early retention and compare their plans, acquisition channels, service quality and complaint history. Improving retention during the first 3–6 months may reduce longer-term churn.

In [53]:
#exporting the cohort-retention table

In [55]:
from pathlib import Path

clean_folder = Path(r"C:\Users\Dell\Nexatel_project\cleaned_data")

cohort_retention_export = (
    cohort_retention
    .reset_index()
    .copy())

cohort_retention_export["acquisition_cohort"] = (
    cohort_retention_export["acquisition_cohort"]
    .astype("string"))

cohort_retention_export.to_csv(
    clean_folder / "cohort_retention.csv",
    index=False)

print("Cohort rows:", len(cohort_retention_export))
print("Columns:", cohort_retention_export.columns.tolist())
print("File saved: cohort_retention.csv")

Cohort rows: 147
Columns: ['acquisition_cohort', '3-Month Retention (%)', '6-Month Retention (%)', '12-Month Retention (%)']
File saved: cohort_retention.csv
